# 04 - Smart Chunking & Metadata-First Retrieval

This notebook demonstrates the **structure-aware chunking** and **metadata-first retrieval** features:

1. Compare generic `SentenceSplitter` vs. `ContractNodeParser` on a sample contract
2. Inspect extracted structural metadata (`section_type`, `has_table`, `clause_number`)
3. Query with `section_type` filter (e.g. payment terms for a specific contract)
4. Query with `has_table` filter (e.g. find all pricing tables)
5. Compare filtered vs. unfiltered retrieval results

**Prerequisites:** Run `01_generate_data.ipynb` and `02_ingestion.ipynb` first.

In [ ]:
import sys
sys.path.insert(0, '../src')
from pathlib import Path

from contract_lens.config import get_settings
from contract_lens.observability import init_observability

settings = get_settings()
init_observability(settings)

## 1. Generic vs. Structure-Aware Chunking

Let's load a single contract and compare how `SentenceSplitter` and `ContractNodeParser` handle it.

In [ ]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter
from contract_lens.ingestion.pipeline import parse_filename_metadata
from contract_lens.ingestion.node_parser import ContractNodeParser

# Load all scanned PDFs
data_path = Path('../data/scans')
reader = SimpleDirectoryReader(input_dir=str(data_path), recursive=False)
documents = reader.load_data()

# Enrich metadata
for doc in documents:
    filename = doc.metadata.get('file_name', '')
    meta = parse_filename_metadata(filename)
    doc.metadata.update(meta)

# Filter to ITSVC-001 base contract only
itsvc_docs = [d for d in documents if d.metadata.get('contract_id') == 'ITSVC-001' and d.metadata.get('source_type') == 'base']
print(f'ITSVC-001 base contract: {len(itsvc_docs)} pages')

In [ ]:
# Generic chunking
generic_splitter = SentenceSplitter(chunk_size=1024, chunk_overlap=128)
generic_nodes = generic_splitter.get_nodes_from_documents(itsvc_docs)

# Structure-aware chunking
contract_parser = ContractNodeParser(chunk_size=1024, chunk_overlap=128)
smart_nodes = contract_parser.get_nodes_from_documents(itsvc_docs)

print(f'Generic SentenceSplitter: {len(generic_nodes)} chunks')
print(f'ContractNodeParser:       {len(smart_nodes)} chunks')
print(f'\n--- Generic chunk 0 (first 200 chars) ---')
print(generic_nodes[0].text[:200])
print(f'\n--- Smart chunk 0 (first 200 chars) ---')
print(smart_nodes[0].text[:200])

## 2. Inspect Structural Metadata

Each smart chunk carries `section_type`, `section_name`, `has_table`, and `clause_number`.

In [ ]:
# Parse all documents
all_smart_nodes = contract_parser.get_nodes_from_documents(documents)

print(f'Total smart chunks: {len(all_smart_nodes)}')
print(f'\n{"section_type":<18} {"has_table":<10} {"clause":<8} {"section_name"}')
print('-' * 80)
for node in all_smart_nodes[:20]:
    m = node.metadata
    print(f'{m.get("section_type", ""):<18} {m.get("has_table", ""):<10} {m.get("clause_number", ""):<8} {m.get("section_name", "")[:50]}')

In [ ]:
# Distribution of section types
from collections import Counter

type_counts = Counter(n.metadata.get('section_type', 'unknown') for n in all_smart_nodes)
table_counts = Counter(n.metadata.get('has_table', 'unknown') for n in all_smart_nodes)

print('Section type distribution:')
for st, count in type_counts.most_common():
    print(f'  {st:<18} {count}')

print(f'\nChunks with tables: {table_counts.get("true", 0)}')
print(f'Chunks without:     {table_counts.get("false", 0)}')

## 3. Metadata-Filtered Retrieval: section_type

Query with `section_type="payment"` to find only payment-related chunks.

In [ ]:
from contract_lens.retrieval.query_engine import query_contracts

# Without filter
answer_unfiltered = query_contracts(
    settings,
    question='What are the current hourly rates for ITSVC-001?',
    contract_id='ITSVC-001',
)
print('=== WITHOUT section_type filter ===')
print(answer_unfiltered)

In [ ]:
# With section_type=payment filter
answer_filtered = query_contracts(
    settings,
    question='What are the current hourly rates for ITSVC-001?',
    contract_id='ITSVC-001',
    section_type='payment',
)
print('=== WITH section_type=payment filter ===')
print(answer_filtered)

## 4. Metadata-Filtered Retrieval: has_table

Find chunks that contain tables (pricing schedules, SLA metrics, etc.).

In [ ]:
answer_tables = query_contracts(
    settings,
    question='What pricing tables exist across all contracts?',
    has_table=True,
)
print('=== has_table=true ===')
print(answer_tables)

## 5. Combining Filters

Combine contract_id + section_type + source_type for precision retrieval.

In [ ]:
# SLA penalty terms from amendments only
answer_sla_penalties = query_contracts(
    settings,
    question='What are the current SLA penalty terms and service credits?',
    contract_id='SLA-001',
    section_type='penalties',
)
print('=== SLA-001 penalties ===')
print(answer_sla_penalties)

In [ ]:
# Lease termination terms (PL)
answer_lease_term = query_contracts(
    settings,
    question='Jakie sa warunki wypowiedzenia umowy najmu LEASE-001?',
    contract_id='LEASE-001',
    section_type='termination',
    language='pl',
)
print('=== LEASE-001 termination (PL) ===')
print(answer_lease_term)

## Summary

| Feature | Description |
|---------|-------------|
| `ContractNodeParser` | Splits at section boundaries, preserves clause context |
| `section_type` filter | Controlled vocabulary: scope, payment, termination, confidentiality, liability, sla, penalties, annex, general |
| `has_table` filter | Find chunks containing pricing tables, SLA metrics, etc. |
| `clause_number` filter | Target a specific clause (e.g. "3.1") |
| Combined filters | Stack multiple filters for precision retrieval |

See [docs/smart-chunking.md](../docs/smart-chunking.md) for full documentation.